<a href="https://colab.research.google.com/github/AnIsAsPe/Filtrado__Colaborativo/blob/main/Notebooks/Evaluaci%C3%B3n_y_prediccion_sistema_recomendacion_Netflix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas

In [1]:
!pip install surprise

In [2]:
import pandas as pd
import numpy as np
import random

import psutil
from surprise import Reader, Dataset, SVD
from surprise import dump, accuracy
from surprise.model_selection import cross_validate

In [3]:

def mostrar_ram(paso):
    ram_libre = psutil.virtual_memory().available / (1024.0 ** 3)
    print(f"[{paso}] RAM Libre estimada: {ram_libre:.2f} GB")
mostrar_ram("Inicio")

[Inicio] RAM Libre estimada: 11.54 GB


# Evaluación

Necesitamos cargar el modelo preentrenado y cargar el conjunto de prueba

In [4]:
ruta_modelo = '/content/drive/MyDrive/Modelos/Netflix/modelo_svd_produccion.pkl'
_,svd_completo = dump.load(ruta_modelo)

mostrar_ram("Despues de cargar el modelo entrenado")

[Despues de cargar el modelo entrenado] RAM Libre estimada: 6.47 GB


In [5]:
# Leemos el archivo evaluando fila por fila.
# Podríamoi > 0 asegura que NUNCA saltemos la fila 0 (los encabezados de las columnas).
test_df = pd.read_csv(
    '/content/drive/MyDrive/Datos/Netflix/test_completo.csv',
    # skiprows=lambda i: i > 0 and random.random() > 0.25
)

print(f"Registros cargados exitosamente a la RAM: {len(test_df):,}")
mostrar_ram("Después de leer df_test")

Registros cargados exitosamente a la RAM: 4,454,041
[Después de leer df_test] RAM Libre estimada: 6.38 GB


In [7]:
%%time
# Convertimos al formato requerido por Surprise
testset_clase = list(test_df[['CustomerID', 'MovieID', 'Rating']].itertuples(index=False, name=None))

# Evaluamos en segundos
predicciones_clase = svd_completo.test(testset_clase)
rmse = accuracy.rmse(predicciones_clase, verbose=True)
mae = accuracy.mae(predicciones_clase, verbose=True)

RMSE: 0.8848
MAE:  0.6865
CPU times: user 49.1 s, sys: 1.7 s, total: 50.8 s
Wall time: 51.6 s


# Sistema de Recomendación

Necesitamos un dataset con la información de las películas en el modelo.

In [19]:
# Leemos directamente el archivo en el que eliminamos el ruido en el notbook de entrenamiento
df_trim = pd.read_csv("/content/drive/MyDrive/Datos/Netflix/historial_limpio.csv")
#Extraemos los IDs únicos que el modelo sí conoce
peliculas_entrenadas = df_trim['MovieID'].unique()
len(peliculas_entrenadas)

4055

In [33]:
ruta_titulos = "/content/drive/MyDrive/Datos/Netflix/movie_titles.csv"


df_movies = pd.read_csv(
    ruta_titulos,
    encoding='ISO-8859-1',
    names=["MovieID", "Year", "Title"],
    on_bad_lines="skip"
)
df_movies["MovieID"] = df_movies["MovieID"].astype(int)
df_movies.set_index("MovieID", inplace=True)

# Filtramos el catálogo para dejar SOLO las películas consideradas en el entrenamiento
df_movies = df_movies[df_movies.index.isin(peliculas_entrenadas)]

mostrar_ram("Después de cargar los títulos")

[Después de cargar los títulos] RAM Libre estimada: 3.79 GB


In [35]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3980 entries, 1 to 4499
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Year    3979 non-null   float64
 1   Title   3980 non-null   object 
dtypes: float64(1), object(1)
memory usage: 93.3+ KB


In [29]:
df_movies.Year.describe()

,Year
count,17427.000000
mean,1990.295633
std,16.489299
min,1896.000000
25%,1985.000000
50%,1997.000000
75%,2002.000000
max,2005.000000


# Sistema de Recomendación

In [44]:
def obtener_recomendaciones(user_id, modelo, df_historial, df_catalogo, top_n=10):
    """
    Genera las mejores recomendaciones de películas no vistas para un usuario.
    """
    # --- PARTE 1: EL PERFIL DEL USUARIO ---
    print(f"LAS 10 PELÍCULAS FAVORITAS DEL USUARIO {user_id}")

    # Filtramos el historial solo para este usuario
    historial_usuario = df_historial[df_historial['CustomerID'] == user_id]

    # Ordenamos por calificación de mayor a menor y tomamos las mejores 10
    top_vistas = historial_usuario.sort_values(by='Rating', ascending=False).head(10)

    # Cruzamos con el catálogo para obtener los nombres
    favoritas = []
    for _, row in top_vistas.iterrows():
        id_peli = row['MovieID']
        titulo = df_catalogo.loc[id_peli, 'Title']
        año = df_catalogo.loc[id_peli, 'Year']
        año = "Desconocido" if pd.isna(año) else int(año)

        favoritas.append({
            'Película': titulo,
            'Año': año,
            'Calificación Real': row['Rating']
        })

    display(pd.DataFrame(favoritas))

    # --- PARTE 2: LAS RECOMENDACIONES DEL MODELO ---

    # Obtener todas las películas únicas en el sistema
    todas_las_peliculas = df_catalogo.index.unique()

    # Identificar qué películas YA vio este usuario
    peliculas_vistas = df_historial[df_historial['CustomerID'] == user_id]['MovieID'].unique()

    # Filtrar las películas no vistas
    peliculas_no_vistas = np.setdiff1d(todas_las_peliculas, peliculas_vistas)

    # Predecir la calificación para todas las películas no vistas
    predicciones = [
        modelo.predict(user_id, movie_id)
        for movie_id in peliculas_no_vistas
    ]

    # Ordenar las predicciones de mayor a menor estimación (.est)
    predicciones_ordenadas = sorted(predicciones, key=lambda x: x.est, reverse=True)

    # Tomar el Top N y cruzar con los títulos
    top_predicciones = predicciones_ordenadas[:top_n]

    recomendaciones = []
    for pred in top_predicciones:
        titulo = df_catalogo.loc[pred.iid, 'Title']
        año = df_catalogo.loc[pred.iid, 'Year']



        # Si el año es nulo, mostramos "Desconocido", de lo contrario mostramos el entero
        año = "Desconocido" if pd.isna(año) else int(año)
        recomendaciones.append({
            'Película': titulo,
            'Año': año,
            'Calificación Estimada': round(pred.est, 2)
        })

    return  pd.DataFrame(recomendaciones)

In [49]:
# Elegimos un usuario al azar de tu muestra de test
usuario = test_df['CustomerID'].sample(1).iloc[0]

# Usamos la función de recomendacion
recomendaciones = obtener_recomendaciones(
    user_id=usuario,
    modelo=svd_completo,
    df_historial=df_trim,
    df_catalogo=df_movies,
    top_n=10
)

# Mostramos los resultados
print(f"\n\nPELÍCULAS RECOMENDADAS PARA EL USUARIO {usuario}:")
display(recomendaciones)

LAS 10 PELÍCULAS FAVORITAS DEL USUARIO 474226


,Película,Año,Calificación Real
0,Reservoir Dogs,1992,5
1,Rambo: First Blood Part II,1985,5
2,Monsoon Wedding,2001,5
3,Clerks,1994,5
4,Taxi,2004,5
5,House of Sand and Fog,2003,5
6,Pay It Forward,2000,5
7,Chasing Amy,1997,5
8,High Fidelity,2000,5
9,Harold and Kumar Go to White Castle,2004,5




PELÍCULAS RECOMENDADAS PARA EL USUARIO 474226:


,Película,Año,Calificación Estimada
0,As Time Goes By: Series 8,2000,5
1,Eternal Sunshine of the Spotless Mind,2004,5
2,Samurai Champloo,2004,5
3,Firefly,2002,5
4,The Simpsons: Season 3,1991,5
5,Prime Suspect 1,1991,5
6,Family Guy: Freakin' Sweet Collection,2004,5
7,Lost: Season 1,2004,5
8,Curb Your Enthusiasm: Season 3,2002,5
9,The West Wing: Season 3,2001,5


In [50]:
# Elegimos un usuario al azar de tu muestra de test
usuario = test_df['CustomerID'].sample(1).iloc[0]

# Usamos la función de recomendacion
recomendaciones = obtener_recomendaciones(
    user_id=usuario,
    modelo=svd_completo,
    df_historial=df_trim,
    df_catalogo=df_movies,
    top_n=10
)

# Mostramos los resultados
print(f"\n\nPELÍCULAS RECOMENDADAS PARA EL USUARIO {usuario}:")
display(recomendaciones)

LAS 10 PELÍCULAS FAVORITAS DEL USUARIO 1628409


,Película,Año,Calificación Real
0,Black Orpheus,1959,5
1,The Chorus,2004,4
2,Rabbit-Proof Fence,2002,4
3,House of Sand and Fog,2003,4
4,Man on the Train,2002,4
5,A Beautiful Mind,2001,4
6,The Pianist,2002,4
7,Secondhand Lions,2003,4
8,Whale Rider,2003,4
9,The Last Samurai,2003,4




PELÍCULAS RECOMENDADAS PARA EL USUARIO 1628409:


,Película,Año,Calificación Estimada
0,Les Miserables in Concert,1996,4.34
1,Babylon 5: Season 4,1996,4.33
2,The Rise and Fall of ECW,2004,4.31
3,NYPD Blue: Season 2,1994,4.29
4,The West Wing: Season 3,2001,4.28
5,I Love Lucy: Season 5,1955,4.23
6,Cracker: Series 2,1994,4.17
7,Lost: Season 1,2004,4.15
8,Farscape: Season 2,2000,4.14
9,Little House on the Prairie: Season 6,1979,4.14


Arriba de este algoritmo, tendríamos que poner reglas de negocio para asegurar por ejemplo que el sistema no recomiende temporadas avanzadas de una serie si el usuario no ha visto las anteriores.